# A1 · Ejercicio de embeddings manuales
**LSCA2318 · Tema 2.** Antes de que un modelo calcule vectores, tú los vas a construir a mano para entender qué significa «parecido = cerca».

**Qué haces:**
1. Eliges **4 a 6 dimensiones con nombre** que tengan sentido en tu variante (ej. `seguridad, pruebas, dinero, despliegue`).
2. Escribes **3 frases** de tu variante: F1 y F2 del mismo tema, F3 de otro tema.
3. Les das un valor de 0 a 1 a cada dimensión (justifícalo en una línea).
4. Normalizas y calculas el coseno F1–F2 y F1–F3. **Interpretas** el resultado.

Funciona en Colab o en VS Code. No necesita internet ni llave.

In [1]:
import numpy as np

# 1) Dimensiones de v3-soporte (soporte a desarrolladores de socios)
dimensiones = ["autenticacion", "transferencias", "errores", "webhooks"]

# 2) y 3) Frases y vectores (0 a 1 por dimensión, mismo orden)
frases = {
    "F1": ("Mi token de acceso expiró y recibo 401",                    [0.9, 0.1, 0.6, 0.0]),
    "F2": ("Cómo renuevo mi credencial OAuth si el token venció",       [0.95, 0.05, 0.2, 0.0]),
    "F3": ("Cómo verifico la firma de un evento webhook de FinNorte",   [0.2, 0.2, 0.0, 0.9]),
}
for k, (texto, v) in frases.items():
    assert len(v) == len(dimensiones), f"{k} debe tener {len(dimensiones)} valores"
    print(k, texto, dict(zip(dimensiones, v)))


F1 Mi token de acceso expiró y recibo 401 {'autenticacion': 0.9, 'transferencias': 0.1, 'errores': 0.6, 'webhooks': 0.0}
F2 Cómo renuevo mi credencial OAuth si el token venció {'autenticacion': 0.95, 'transferencias': 0.05, 'errores': 0.2, 'webhooks': 0.0}
F3 Cómo verifico la firma de un evento webhook de FinNorte {'autenticacion': 0.2, 'transferencias': 0.2, 'errores': 0.0, 'webhooks': 0.9}


**Justificación de tus valores:**

- F1: «token» y «expiró» son autenticación pura (0.9); el 401 es un código de error (0.6); no habla de transferencias (0.1) ni de webhooks (0).
- F2: «credencial OAuth» y «token venció» son autenticación casi total (0.95); solo menciona el problema de forma indirecta (errores 0.2); sin transferencias ni webhooks.
- F3: es sobre webhooks y firma de eventos (0.9); la firma toca algo de autenticación/seguridad (0.2) y de transferencias (0.2) porque los eventos son de transferencias; no es un error (0).


In [3]:
def normalizar(v):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)
    return v / n if n > 0 else v      # el vector cero se queda en cero

def coseno(a, b):
    a, b = normalizar(a), normalizar(b)
    return float(a @ b)

V = {k: v for k, (_, v) in frases.items()}
for k in V:
    print(f"{k} normalizado:", np.round(normalizar(V[k]), 3), " norma =", round(np.linalg.norm(normalizar(V[k])), 3))

print(f"\ncos(F1, F2) = {coseno(V['F1'], V['F2']):.3f}   ← mismo tema")
print(f"cos(F1, F3) = {coseno(V['F1'], V['F3']):.3f}   ← temas distintos")

F1 normalizado: [0.829 0.092 0.552 0.   ]  norma = 1.0
F2 normalizado: [0.977 0.051 0.206 0.   ]  norma = 1.0
F3 normalizado: [0.212 0.212 0.    0.954]  norma = 1.0

cos(F1, F2) = 0.928   ← mismo tema
cos(F1, F3) = 0.195   ← temas distintos


**Interpretación:**
1. Sí: cos(F1, F2) ≈ 0.93 es mucho mayor que cos(F1, F3) ≈ 0.20. F1 y F2 apuntan casi en la misma dirección porque ambas son sobre autenticación; F3 apunta hacia «webhooks», otra dirección.
2. La dimensión que más pesó fue «autenticacion»: F1 y F2 tienen valores altos ahí (0.9 y 0.95) y el producto punto se concentra en esa dimensión.
3. Con dimensiones con nombre puedo explicar y auditar por qué dos frases se parecen, pero yo mismo las asigno a mano y solo captan 4 temas. Un modelo real usa cientos de dimensiones sin nombre aprendidas de millones de textos: capta sinónimos y matices que yo no preví, pero pierdo la interpretabilidad (no sé qué significa cada dimensión).
